## Harvested Articles

In [4]:
import sys
from pathlib import Path
import pandas as pd

import json
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

from tqdm import tqdm
import re

project_root = Path("../").resolve()
sys.path.insert(0, str(project_root))

from utils.evals import create_table_total_articles

## Original Files from PERG
- irrelevant -> Removed during Abstract Screening 
- excluded -> Removed during Full-text Review
- included -> Considered for final Full-text Review

In [4]:
pathogens = ['Lassa','Ebola','MERS','Nipah','SARS','Zika','Marburg']
files = ['01_pathogen_irrelevant_csv.csv','02_pathogen_excluded_csv.csv','03_pathogen_included_csv.csv']

directory = str(project_root / 'data' / 'perg')

df_array = []

for p in pathogens:
    for f in files:
        df_1 = pd.read_csv(directory + '/screening_raw/' + f'01_{p}_irrelevant_csv.csv')
        df_2 = pd.read_csv(directory + '/screening_raw/' + f'02_{p}_excluded_csv.csv')
        df_3 = pd.read_csv(directory + '/screening_raw/' + f'03_{p}_included_csv.csv')

        df_1['perg_abstract_result'] =  'EXCLUDE'
        df_2['perg_abstract_result'] =  'INCLUDE'
        df_3['perg_abstract_result'] =  'INCLUDE'

        df_1['perg_fulltext_result'] =  'EXCLUDE'
        df_2['perg_fulltext_result'] =  'EXCLUDE'
        df_3['perg_fulltext_result'] =  'INCLUDE'

        df_perg = pd.concat([df_1,df_2,df_3])
        df_perg.to_csv(f'{directory}/screening/{p}_filtered.csv')

## Mapping Downloaded articles to PERG's set (Table 1)

In [ ]:
pathogens = ["Marburg", "Ebola", "Lassa", "SARS", "Zika", "Nipah", "MERS", "RVF", "CCHF"]

df_summary = create_table_total_articles.build_summary(
    pathogens=pathogens,
    perg_template="../data/perg/screening/{pathogen}_filtered.csv",
    harvest_template_regular="../data/agentslr/harvests/{pathogen_lower}/articles_with_markdown.csv",
    harvest_template_rvf_cchf="../data/agentslr/harvests/{pathogen_lower}/articles_with_markdown.csv",
)

Processing pathogens: 100%|██████████| 9/9 [01:11<00:00,  7.92s/it]


In [3]:
df_summary

,Pathogen,PERG Articles (CSVs),PERG Articles (MRC Website),Our Articles (All),Our Articles (PERG Subset)
0,Marburg virus,2593,4460,6501,762 (29.4%)
1,Ebola virus,11605,14690,23226,3938 (33.9%)
2,Lassa Mammarenavirus,2131,2817,6514,647 (30.4%)
3,SARS-CoV-1,12280,14929,7540,1967 (16.0%)
4,Zika virus,10510,4518,3103,2128 (20.2%)
5,Nipah virus,1458,1842,5103,664 (45.5%)
6,MERS-CoV,19656,10382,23204,5675 (28.9%)
7,Rift Valley Fever Virus,-,3341,6810,-
8,Nairo virus (CCHF),-,1967,3478,-


In [4]:
latex = create_table_total_articles.df_summary_to_perg_latex(
    df_summary,
    label="tab:summary_of_PERG_reviews_lean",
)
print(latex)

\begin{table}[t]
\centering
\small
\setlength{\tabcolsep}{3pt}
\renewcommand{\arraystretch}{1.1}
\label{tab:summary_of_PERG_reviews_lean}
\begin{tabular}{lrrr}
\toprule
\textbf{Pathogen} & \textbf{PERG} & \textbf{Ours} & \textbf{Matched to PERG} \\
\midrule
Marburg virus$^{*}$ & 2,593 & 6,501 & 762 (29.4\%) \\
Ebola virus$^{*}$ & 11,605 & 23,226 & 3,938 (33.9\%) \\
Lassa Mammarenavirus$^{*}$ & 2,131 & 6,514 & 647 (30.4\%) \\
SARS-CoV-1$^{*}$ & 12,280 & 7,540 & 1,967 (16.0\%) \\
Zika virus$^{*}$ & 10,510 & 3,103 & 2,128 (20.2\%) \\
Nipah virus & 1,458 & 5,103 & 664 (45.5\%) \\
MERS-CoV & 19,656 & 23,204 & 5,675 (28.9\%) \\
Rift Valley Fever Virus & -- & 6,810 & -- \\
Nairo virus (CCHF) & -- & 3,478 & -- \\
\midrule
\textbf{Total}$^{*}$ & 60,233 & 75,191 & 15,781 (26.2\%) \\
\bottomrule
\end{tabular}
\vspace{2pt}
\par\footnotesize $^{*}$Total excludes RVF and CCHF.
\end{table}


# Articles Present in Each Stage — PERG
* Used for computing `Time taken to conduct SLR for Humans vs AgentSLR` and `Cost vs Performance`.

In [ ]:
pathogens = ["Ebola", "Lassa", "SARS", "Zika"]

screning_path_template = "../data/perg/screening/{pathogen}_filtered.csv"

# Taken from PERG MRC website
perg_website_total_articles = {
    "Ebola": 11605,
    "Lassa": 2131,
    "SARS": 12280,
    "Zika": 10510,
}

articles_passed_through = []

for pathogen in pathogens:
    df = pd.read_csv(screning_path_template.format(pathogen=pathogen))
    articles_passed_through.append({
        'Pathogen': pathogen,
        'Title & Abstract Screening': perg_website_total_articles.get(pathogen, len(df)),
        'Full-text Screening': len(df[df.perg_abstract_result=='INCLUDE']),
        'Data Extraction': len(df[df.perg_fulltext_result=='INCLUDE']),
    })
df_articles_passed = pd.DataFrame(articles_passed_through)
average_row = {
    'Pathogen': 'Average',
    'Title & Abstract Screening': df_articles_passed['Title & Abstract Screening'].mean(),
    'Full-text Screening': df_articles_passed['Full-text Screening'].mean(),
    'Data Extraction': df_articles_passed['Data Extraction'].mean(),
}

In [41]:
df = pd.concat([df_articles_passed, pd.DataFrame([average_row])], ignore_index=True)
df

,Pathogen,Title & Abstract Screening,Full-text Screening,Data Extraction
0,Ebola,11605.0,1674.00,522.0
1,Lassa,2131.0,512.00,193.0
2,SARS,12280.0,878.00,289.0
3,Zika,10510.0,1343.00,574.0
4,Average,9131.5,1101.75,394.5


In [ ]:
df.to_csv("../data/agentslr/evals/stats/articles_passed_through_stages_perg.csv", index=False)

# Token and Cost Uage 

In [15]:

models = ["oss", "deepseek", "kimi", "glm", "gpt"]

stage_specs = [
    ("Title & Abstract Screening", "abstract_screening_dumps_", ["Marburg", "Ebola", "Lassa", "SARS", "Zika", "MERS", "Nipah"]),
    ("Article Screening (AI Conditioned)", "fulltext_screening_dumps_", ["Marburg", "Ebola", "Lassa", "SARS", "Zika", "MERS", "Nipah"]),
    ("Parameter Extraction", "data_extraction_parameters_dumps_", ["Ebola", "Lassa", "SARS", "Zika"]),
    ("Model Extraction", "data_extraction_models_dumps_", ["Ebola", "Lassa", "SARS", "Zika"]),
    ("Outbreak Extraction", "data_extraction_outbreaks_dumps_", ["Lassa", "Zika"]),
]

stage_keys = [s[0] for s in stage_specs]

root = Path("../data/agentslr/client")


def ts_re(prefix):
    return re.compile(rf"^{re.escape(prefix)}(\d{{4}}-\d{{2}}-\d{{2}}_\d{{2}}-\d{{2}}-\d{{2}})$")


def stage_dirs(base, prefix):
    rx = ts_re(prefix)
    out = []
    if base.exists():
        for d in base.iterdir():
            if d.is_dir():
                m = rx.match(d.name)
                if m:
                    out.append((m.group(1), d))
    out.sort(key=lambda x: x[0], reverse=True)
    return out


def trace_candidates(d, pathogen):
    return [
        d / pathogen / "reasoning_traces.jsonl",
        d / pathogen / "results" / "reasoning_traces.jsonl",
        d / "reasoning_traces.jsonl",
        d / "results" / "reasoning_traces.jsonl",
    ]


def find_trace(d, pathogen):
    for p in trace_candidates(d, pathogen):
        if p.is_file():
            return p
    pdir = d / pathogen
    if pdir.exists():
        hits = sorted(pdir.rglob("reasoning_traces.jsonl"))
        if hits:
            return hits[0]
    hits = sorted(d.rglob("reasoning_traces.jsonl"))
    return hits[0] if hits else None

In [16]:


reasoning_traces_paths = {}
reasoning_traces_flags = {}

total = sum(len(ps) * len(models) for _, _, ps in stage_specs)

with tqdm(total=total, desc="Locating reasoning_traces.jsonl") as pbar:
    for model_name in models:
        for stage_key, prefix, pathogens in stage_specs:
            for pathogen in pathogens:
                base = root / model_name / pathogen / "logs"
                candidates = stage_dirs(base, prefix)

                key = (model_name, pathogen)
                reasoning_traces_paths.setdefault(key, {})
                reasoning_traces_flags.setdefault(key, {})

                chosen = None
                missing = []

                if not candidates:
                    reasoning_traces_paths[key][stage_key] = None
                    reasoning_traces_flags[key][stage_key] = {
                        "issue": "MISSING_LOGS_OR_STAGE_DIR",
                        "used_ts": None,
                        "missing_newer": [],
                    }
                    print(f"MISSING LOGS/STAGE DIR: {(model_name, pathogen, stage_key)} base={base}")
                    pbar.update(1)
                    continue

                for ts, d in candidates:
                    hit = find_trace(d, pathogen)
                    if hit is not None and hit.is_file():
                        chosen = hit
                        reasoning_traces_flags[key][stage_key] = {
                            "issue": "OK" if not missing else "FALLBACK_USED",
                            "used_ts": ts,
                            "missing_newer": missing,
                        }
                        break
                    missing.append(ts)

                if chosen is None:
                    reasoning_traces_paths[key][stage_key] = None
                    reasoning_traces_flags[key][stage_key] = {
                        "issue": "MISSING_ALL_CANDIDATES",
                        "used_ts": None,
                        "missing_newer": missing,
                    }
                    print(f"MISSING ALL: {(model_name, pathogen, stage_key)} checked={missing}")
                else:
                    reasoning_traces_paths[key][stage_key] = str(chosen)
                    if missing:
                        print(
                            f"FALLBACK USED: {(model_name, pathogen, stage_key)} "
                            f"used_ts={reasoning_traces_flags[key][stage_key]['used_ts']} "
                            f"missing_newer={missing}"
                        )

                pbar.update(1)

Locating reasoning_traces.jsonl:   0%|          | 0/120 [00:00<?, ?it/s]

Locating reasoning_traces.jsonl: 100%|██████████| 120/120 [00:00<00:00, 739.27it/s]

MISSING LOGS/STAGE DIR: ('oss', 'Marburg', 'Title & Abstract Screening') base=../data/agentslr/client/oss/Marburg/logs
MISSING LOGS/STAGE DIR: ('oss', 'Marburg', 'Article Screening (AI Conditioned)') base=../data/agentslr/client/oss/Marburg/logs
MISSING LOGS/STAGE DIR: ('oss', 'Ebola', 'Article Screening (AI Conditioned)') base=../data/agentslr/client/oss/Ebola/logs
MISSING LOGS/STAGE DIR: ('oss', 'MERS', 'Article Screening (AI Conditioned)') base=../data/agentslr/client/oss/MERS/logs
MISSING LOGS/STAGE DIR: ('oss', 'Nipah', 'Article Screening (AI Conditioned)') base=../data/agentslr/client/oss/Nipah/logs
MISSING LOGS/STAGE DIR: ('oss', 'Ebola', 'Model Extraction') base=../data/agentslr/client/oss/Ebola/logs


In [17]:
def iter_jsonl(path: str):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)

def get_usage_tokens(usage: dict):
    if not usage:
        return 0, 0
    if "input_tokens" in usage or "output_tokens" in usage:
        return int(usage.get("input_tokens") or 0), int(usage.get("output_tokens") or 0)
    return int(usage.get("prompt_tokens") or 0), int(usage.get("completion_tokens") or 0)

def iter_usages(rec: dict):
    resp = rec.get("response") or {}
    u = resp.get("usage") or {}
    if u:
        yield u
    for it in rec.get("iterations") or []:
        r = (it or {}).get("response") or {}
        u2 = r.get("usage") or {}
        if u2:
            yield u2

def compute_row(args):
    rec, group_key = args
    key = rec.get(group_key)
    if key is None:
        return None, 0, 0
    in_tok = 0
    out_tok = 0
    for usage in iter_usages(rec):
        i, o = get_usage_tokens(usage)
        in_tok += i
        out_tok += o
    return key, in_tok, out_tok

def make_df_grouped_jsonl(path: str, group_key: str, workers: int = 16, batch_size: int = 5000):
    acc = {}

    def add_row(row):
        key, inp, outp = row
        if key is None:
            return
        v = acc.get(key)
        if v is None:
            acc[key] = [inp, outp]
        else:
            v[0] += inp
            v[1] += outp

    buf = []
    with ThreadPoolExecutor(max_workers=workers) as ex:
        for rec in iter_jsonl(path):
            buf.append(rec)
            if len(buf) >= batch_size:
                for row in ex.map(compute_row, [(r, group_key) for r in buf]):
                    add_row(row)
                buf.clear()
        if buf:
            for row in ex.map(compute_row, [(r, group_key) for r in buf]):
                add_row(row)

    return (
        pd.DataFrame(
            [(k, v[0], v[1]) for k, v in acc.items()],
            columns=[group_key, "input_tokens", "output_tokens"],
        )
        .sort_values(group_key, ignore_index=True)
    )

In [18]:
df_all = []

for model_name, pathogen in tqdm(reasoning_traces_paths, desc="Processing reasoning traces"):

    file_paths = reasoning_traces_paths[(model_name, pathogen)]
    for stage, path in file_paths.items():
        if path:
            if 'Screening' in stage:
                groupped_key = 'trace_id'
            else:                  
                groupped_key = 'article_id'

            df = make_df_grouped_jsonl(path, group_key=groupped_key, workers=16)

            
            # remove rows that have 0 input / output tokens, as they likely indicate failed runs or missing data
            df = df[(df["input_tokens"] > 0) | (df["output_tokens"] > 0)].copy()

            if len(df) == 0:
                print(f"No valid traces found for model={model_name}, pathogen={pathogen}, stage={stage} at path={path}")
                continue
            
            df['model'] = model_name
            df['stage'] = stage
            df['pathogen'] = pathogen
            df_all.append(df)


        else:
            print(f"Missing path for model={model_name}, pathogen={pathogen}, stage={stage}")

Processing reasoning traces:   0%|          | 0/35 [00:00<?, ?it/s]

Missing path for model=oss, pathogen=Marburg, stage=Title & Abstract Screening
Missing path for model=oss, pathogen=Marburg, stage=Article Screening (AI Conditioned)
Missing path for model=oss, pathogen=Ebola, stage=Article Screening (AI Conditioned)


Processing reasoning traces:   6%|▌         | 2/35 [00:01<00:22,  1.44it/s]

Missing path for model=oss, pathogen=Ebola, stage=Model Extraction


Processing reasoning traces:  17%|█▋        | 6/35 [00:06<00:31,  1.09s/it]

Missing path for model=oss, pathogen=MERS, stage=Article Screening (AI Conditioned)
Missing path for model=oss, pathogen=Nipah, stage=Article Screening (AI Conditioned)


Processing reasoning traces: 100%|██████████| 35/35 [00:23<00:00,  1.50it/s]


In [19]:
df_tokens = pd.concat(df_all, ignore_index=True)

In [20]:
df_tokens.to_csv("../data/agentslr/evals/stats/token_usage.csv", index=False)

In [21]:
df_tokens.groupby(["model", "stage"])[["input_tokens", "output_tokens"]].mean().reset_index()

,model,stage,input_tokens,output_tokens
0,deepseek,Article Screening (AI Conditioned),1.285625e+04,837.294788
1,deepseek,Model Extraction,3.724775e+04,3141.173507
2,deepseek,Outbreak Extraction,2.610066e+04,209.509524
3,deepseek,Parameter Extraction,5.232496e+05,2989.410448
4,deepseek,Title & Abstract Screening,2.168332e+03,614.413497
5,glm,Article Screening (AI Conditioned),1.338633e+04,3120.621041
6,glm,Model Extraction,2.973981e+04,1673.203358
7,glm,Outbreak Extraction,2.537823e+04,1980.566667
8,glm,Parameter Extraction,3.050365e+06,32671.718284
9,glm,Title & Abstract Screening,2.224815e+03,1644.283988


### Time Statistitics

In [2]:
import re
from pathlib import Path
import pandas as pd
import tqdm

In [3]:
pathogens = ["Ebola", "Lassa", "SARS", "Zika"]

root = Path("../data/agentslr/client/deepseek")
prefixes = [
    "data_extraction_outbreaks_dumps_",
    "data_extraction_models_dumps_",
]

# match: data_extraction_outbreaks_dumps_YYYY-MM-DD_HH-MM-SS
# match: data_extraction_models_dumps_YYYY-MM-DD_HH-MM-SS
ts_res = {
    p: re.compile(rf"^{re.escape(p)}(\d{{4}}-\d{{2}}-\d{{2}}_\d{{2}}-\d{{2}}-\d{{2}})$")
    for p in prefixes
}

# {(pathogen, kind): "/path/to/screening.jsonl" or None}
data_extraction_agentslr = {}

for pathogen in pathogens:
    base = root / pathogen / "logs"

    for prefix in prefixes:
        candidates = []

        if base.exists():
            ts_re = ts_res[prefix]
            for d in base.iterdir():
                if d.is_dir():
                    m = ts_re.match(d.name)
                    if m:
                        candidates.append((m.group(1), d))

        candidates.sort(key=lambda x: x[0], reverse=True)

        chosen = None
        missing = []

        for ts, d in candidates:
            target = d / "reasoning_traces.jsonl"
            if target.is_file():
                chosen = target
                break
            missing.append(ts)

        key = (pathogen, prefix)

        if chosen is None:
            data_extraction_agentslr[key] = None
            if candidates:
                print(f"MISSING ALL: {key} checked={[ts for ts, _ in candidates]}")
            else:
                print(f"MISSING LOGS DIR: {key}")
        else:
            data_extraction_agentslr[key] = str(chosen)
            if missing:
                print(
                    f"FALLBACK USED: {key} "
                    f"used={chosen.parent.parent.name} "
                    f"missing_newer={missing}"
                )

In [33]:
'''
Total unique documents: 103261
  Total pages: 1714176
  Average pages per document: 16.60
  Min pages: 1
  Max pages: 849

With parallelism (14 requests in parallel):
  Avg time per page (parallel):     0.06s
  Avg time per document (parallel): 1.10s


Without (running sequentially):
Avg time per page (sequential):   0.95s
Avg time per document (sequential): 16.47s

'''

def processs_time(df, time_column):
    max_time = pd.to_datetime(df[time_column].max())
    min_time = pd.to_datetime(df[time_column].min())
    time_per_article_minutes = (max_time - min_time) / len(df) / pd.Timedelta(minutes=1)
    return time_per_article_minutes

def compute_one(model, pathogen):
    df_articles = pd.read_csv(f'../data/agentslr/harvests/{pathogen.lower()}/articles_with_markdown.csv')
    df_abstract_screening = pd.read_csv(f'../data/agentslr/client/{model}/{pathogen}/screening/abstract_screening_results.csv')
    df_fulltext_screening = pd.read_csv(f'../data/agentslr/client/{model}/{pathogen}/screening/fulltext_screening_results.csv')
    df_parameters = pd.read_json(
        f'../data/agentslr/client/{model}/{pathogen}/extractions/data_extraction_parameters.jsonl',
        lines=True
    )

    time_per_article_minutes_abstract = processs_time(df_abstract_screening, 'screening_timestamp')
    
    try:
        time_per_article_minutes_fulltext = processs_time(df_fulltext_screening, 'screening_timestamp')
    except KeyError:
        time_per_article_minutes_fulltext = processs_time(df_fulltext_screening, 'fulltext_timestamp')

    time_per_article_minutes_parameters = processs_time(df_parameters, 'timestamp')

    time_per_article_minutes_harvest = processs_time(df_articles, 'harvested_at')
    time_per_article_minutes_download = processs_time(df_articles, 'download_attempted_at') + time_per_article_minutes_harvest

    if model !='deepseek':
        df_outbreaks = pd.read_csv(f'../data/agentslr/client/{model}/{pathogen}/extractions/data_extraction_outbreaks.csv')
        time_per_article_minutes_outbreaks = processs_time(df_outbreaks, 'extraction_timestamp')

        df_models = pd.read_csv(f'../data/agentslr/client/{model}/{pathogen}/extractions/data_extraction_models.csv')
        time_per_article_minutes_models = processs_time(df_models, 'extraction_timestamp')
        
    if model =='deepseek':
        outbreaks_path = data_extraction_agentslr[(pathogen, 'data_extraction_outbreaks_dumps_')]
        models_path = data_extraction_agentslr[(pathogen, 'data_extraction_models_dumps_')]
        df_outbreaks = pd.read_json(outbreaks_path, lines=True)
        df_models = pd.read_json(models_path, lines=True)
        time_per_article_minutes_outbreaks = processs_time(df_outbreaks, 'timestamp')
        time_per_article_minutes_models = processs_time(df_models, 'timestamp')

    return (model, pathogen), {
        'Title & Abstract Screening': time_per_article_minutes_abstract,
        'Download and Harvest': time_per_article_minutes_download,
        'Full-text Screening': time_per_article_minutes_fulltext,
        'Parameter Extraction': time_per_article_minutes_parameters,
        'Model Extraction': time_per_article_minutes_models,
        'Outbreak Extraction': time_per_article_minutes_outbreaks,
        'PDF-to-MD Conversion': 1.1/60 # Mistral API Parallell processing 14 requests (16.6 pages per avg PDF)
    }

In [34]:
models = ['oss','gpt', 'glm', 'kimi', 'deepseek']
pathogens = ['Ebola', 'Lassa', 'SARS', 'Zika']
data = {}

for model in tqdm.tqdm(models):
    for pathogen in pathogens:
        key, value = compute_one(model, pathogen)
        print(f"{key}: {value}")
        data[key] = value

  0%|          | 0/5 [00:00<?, ?it/s]

('oss', 'Ebola'): {'Title & Abstract Screening': 0.009943216666666666, 'Download and Harvest': 0.010128966666666666, 'Full-text Screening': 0.0293619, 'Parameter Extraction': 1.4849773333333334, 'Model Extraction': 0.15753191666666666, 'Outbreak Extraction': 0.10961711666666667, 'PDF-to-MD Conversion': 0.018333333333333333}
('oss', 'Lassa'): {'Title & Abstract Screening': 0.014402183333333334, 'Download and Harvest': 0.017335183333333334, 'Full-text Screening': 0.027789916666666668, 'Parameter Extraction': 2.621231933333333, 'Model Extraction': 0.20437451666666667, 'Outbreak Extraction': 0.14270053333333332, 'PDF-to-MD Conversion': 0.018333333333333333}
('oss', 'SARS'): {'Title & Abstract Screening': 0.0072002, 'Download and Harvest': 0.011868266666666667, 'Full-text Screening': 0.028799283333333335, 'Parameter Extraction': 1.4156693833333334, 'Model Extraction': 0.15537458333333334, 'Outbreak Extraction': 0.11882433333333334, 'PDF-to-MD Conversion': 0.018333333333333333}


 20%|██        | 1/5 [01:35<06:20, 95.19s/it]

('oss', 'Zika'): {'Title & Abstract Screening': 0.010170933333333333, 'Download and Harvest': 0.0029899166666666668, 'Full-text Screening': 0.049333683333333336, 'Parameter Extraction': 1.3561194833333334, 'Model Extraction': 0.19323806666666668, 'Outbreak Extraction': 0.17937533333333333, 'PDF-to-MD Conversion': 0.018333333333333333}
('gpt', 'Ebola'): {'Title & Abstract Screening': 0.02829555, 'Download and Harvest': 0.010128966666666666, 'Full-text Screening': 0.20968125, 'Parameter Extraction': 0.69966545, 'Model Extraction': 1.1223038333333333, 'Outbreak Extraction': 1.3310018333333333, 'PDF-to-MD Conversion': 0.018333333333333333}
('gpt', 'Lassa'): {'Title & Abstract Screening': 0.02680455, 'Download and Harvest': 0.017335183333333334, 'Full-text Screening': 0.14676583333333335, 'Parameter Extraction': 7.240597583333333, 'Model Extraction': 1.4522546166666668, 'Outbreak Extraction': 1.2468316, 'PDF-to-MD Conversion': 0.018333333333333333}
('gpt', 'SARS'): {'Title & Abstract Screen

 40%|████      | 2/5 [02:11<03:01, 60.54s/it]

...

('glm', 'Zika'): {'Title & Abstract Screening': 0.054694283333333336, 'Download and Harvest': 0.0029899166666666668, 'Full-text Screening': 0.27499495, 'Parameter Extraction': 2.7154811333333333, 'Model Extraction': 1.0345502, 'Outbreak Extraction': 1.4555062833333334, 'PDF-to-MD Conversion': 0.018333333333333333}
('kimi', 'Ebola'): {'Title & Abstract Screening': 0.037428583333333335, 'Download and Harvest': 0.010128966666666666, 'Full-text Screening': 0.24107048333333334, 'Parameter Extraction': 2.07084315, 'Model Extraction': 0.55363785, 'Outbreak Extraction': 0.5849432666666666, 'PDF-to-MD Conversion': 0.018333333333333333}
('kimi', 'Lassa'): {'Title & Abstract Screening': 0.03489953333333334, 'Download and Harvest': 0.017335183333333334, 'Full-text Screening': 0.18213196666666667, 'Parameter Extraction': 1.8158413666666666, 'Model Extraction': 0.5426184, 'Outbreak Extraction': 0.4346948666666667, 'PDF-to-MD Conversion': 0.018333333333333333}
('kimi', 'SARS'): {'Title & Abstract Scr

 80%|████████  | 4/5 [03:27<00:45, 45.21s/it]

('kimi', 'Zika'): {'Title & Abstract Screening': 0.0433188, 'Download and Harvest': 0.0029899166666666668, 'Full-text Screening': 0.19317865, 'Parameter Extraction': 1.5616053333333333, 'Model Extraction': 0.6244489333333333, 'Outbreak Extraction': 0.48679543333333336, 'PDF-to-MD Conversion': 0.018333333333333333}
('deepseek', 'Ebola'): {'Title & Abstract Screening': 0.03912673333333333, 'Download and Harvest': 0.010128966666666666, 'Full-text Screening': 0.31848238333333334, 'Parameter Extraction': 0.3344740666666667, 'Model Extraction': 0.7515564, 'Outbreak Extraction': 0.10023731666666667, 'PDF-to-MD Conversion': 0.018333333333333333}
('deepseek', 'Lassa'): {'Title & Abstract Screening': 0.03614771666666667, 'Download and Harvest': 0.017335183333333334, 'Full-text Screening': 0.2136872, 'Parameter Extraction': 0.311115, 'Model Extraction': 0.6419403333333333, 'Outbreak Extraction': 0.08658573333333333, 'PDF-to-MD Conversion': 0.018333333333333333}
('deepseek', 'SARS'): {'Title & Abs

100%|██████████| 5/5 [04:03<00:00, 48.74s/it]

('deepseek', 'Zika'): {'Title & Abstract Screening': 0.041409016666666666, 'Download and Harvest': 0.0029899166666666668, 'Full-text Screening': 0.2165745, 'Parameter Extraction': 0.30995043333333333, 'Model Extraction': 0.4604026, 'Outbreak Extraction': 0.0629749, 'PDF-to-MD Conversion': 0.018333333333333333}


In [35]:
df = pd.DataFrame.from_dict(data, orient='index').reset_index().rename(columns={'index': 'Model_Pathogen', 'level_0': 'Model', 'level_1': 'Pathogen'})

In [36]:
df

,Model,Pathogen,Title & Abstract Screening,Download and Harvest,Full-text Screening,Parameter Extraction,Model Extraction,Outbreak Extraction,PDF-to-MD Conversion
0,oss,Ebola,0.009943,0.010129,0.029362,1.484977,0.157532,0.109617,0.018333
1,oss,Lassa,0.014402,0.017335,0.027790,2.621232,0.204375,0.142701,0.018333
2,oss,SARS,0.007200,0.011868,0.028799,1.415669,0.155375,0.118824,0.018333
3,oss,Zika,0.010171,0.002990,0.049334,1.356119,0.193238,0.179375,0.018333
4,gpt,Ebola,0.028296,0.010129,0.209681,0.699665,1.122304,1.331002,0.018333
5,gpt,Lassa,0.026805,0.017335,0.146766,7.240598,1.452255,1.246832,0.018333
6,gpt,SARS,0.031400,0.011868,0.228041,0.849859,1.179945,0.899678,0.018333
7,gpt,Zika,0.032849,0.002990,0.159784,0.651243,1.363787,1.470472,0.018333
8,glm,Ebola,0.047774,0.010129,0.390408,3.621694,0.819372,0.958451,0.018333
9,glm,Lassa,0.047975,0.017335,0.281119,4.000723,1.298073,0.992064,0.018333


In [37]:
df.to_csv("../data/agentslr/evals/stats/time_per_article_minutes.csv", index=False)

### Appendix Section for Time Stats

In [17]:
import pandas as pd

df = pd.read_csv('../data/agentslr/evals/stats/time_per_article_minutes.csv')

In [37]:

df_oss = df[df.Model=='oss']

# Average time across patogens and convert mins to seconds
df_oss_avg = df_oss.mean(numeric_only=True) * 60
df_oss_avg['Model'] = 'oss'

In [ ]:
# Add up Model Extraction, Outbreak Extraction, Parameter Extraction and have Data Extraction as their sum only
df_oss_avg['Data Extraction'] = df_oss_avg['Model Extraction'] + df_oss_avg['Outbreak Extraction'] + df_oss_avg['Parameter Extraction']

# Remove the individual extraction times as they are now represented in the summed 'Data Extraction' column
df_oss_avg = df_oss_avg.drop(['Model Extraction', 'Outbreak Extraction', 'Parameter Extraction'])

# Rename Download and Harvest to Article Retrieval
df_oss_avg = df_oss_avg.rename({
    'Download and Harvest': 'Article Retrieval',
    'PDF-to-MD Conversion': 'PDF-to-Markdown Conversion',
})

df_oss_avg

# multiply df_oss_avg by 9132, 9132 for article retrieval and title and abstract screening | 1,102 for pdf-to-md and full-text screening | 395 for data extraction

df_oss_multiplied_hours = df_oss_avg.copy()

df_oss_multiplied_hours['Title & Abstract Screening'] = df_oss_avg['Title & Abstract Screening'] * 9132
df_oss_multiplied_hours['Article Retrieval'] = df_oss_avg['Article Retrieval'] * 9132
df_oss_multiplied_hours['PDF-to-Markdown Conversion'] = df_oss_avg['PDF-to-Markdown Conversion'] * 9132
df_oss_multiplied_hours['Full-text Screening'] = df_oss_avg['Full-text Screening'] * 1102
df_oss_multiplied_hours['Data Extraction'] = df_oss_avg['Data Extraction'] * 395



In [20]:
df_oss_avg

Title & Abstract Screening      0.625748
Article Retrieval               0.634835
Full-text Screening             2.029272
PDF-to-Markdown Conversion           1.1
Model                                oss
Data Extraction               122.085518
dtype: object

In [23]:
df_oss_multiplied_hours

Title & Abstract Screening    5714.330736
Article Retrieval              5797.31322
Full-text Screening           2236.257468
PDF-to-Markdown Conversion         1212.2
Model                                 oss
Data Extraction               48223.77961
dtype: object

In [39]:
# df_oss_multiplied_hours is in seconds, convert to hours by dividing by 3600, model column is str, keep it as is
# drop "Model" column
df_oss_multiplied_hours = df_oss_multiplied_hours.drop('Model', errors='ignore')

df_oss_multiplied_hours = df_oss_multiplied_hours / 3600
df_oss_multiplied_hours

Title & Abstract Screening     1.587314
Article Retrieval              1.610365
Full-text Screening            0.621183
PDF-to-Markdown Conversion     0.336722
Data Extraction               13.395494
dtype: object

In [40]:
sum(df_oss_multiplied_hours)

17.55107806513887

In [38]:
# Average articles passed through each stage.
t_and_a = 9132
full_text = 1102
extraction = 395

In [39]:
average_df = df.groupby('Model').mean(numeric_only=True).reset_index()

time_df = (
    average_df
    .assign(
        **{
            'Title & Abstract Screening': lambda d: d['Title & Abstract Screening'] * t_and_a / 60,
            'Full-text Screening': lambda d: d['Full-text Screening'] * full_text / 60,
            'PDF-to-MD Conversion': lambda d: d['PDF-to-MD Conversion'] * full_text / 60,
            'Data Extraction': lambda d: (
                d['Model Extraction'] +
                d['Parameter Extraction'] +
                d['Outbreak Extraction']
            ) * extraction / 60
        }
    )
    [['Model', 'Title & Abstract Screening', 'Full-text Screening', 'PDF-to-MD Conversion', 'Data Extraction']]
    .assign(Total=lambda d: d.iloc[:, 1:].sum(axis=1))
)

In [40]:
time_df

,Model,Title & Abstract Screening,Full-text Screening,PDF-to-MD Conversion,Data Extraction,Total
0,deepseek,5.859977,4.757469,0.336722,7.354723,18.308891
1,glm,7.576480,6.073388,0.336722,37.329672,51.316262
2,gpt,4.541231,3.417451,0.336722,32.106323,40.401728
3,kimi,5.803884,3.910801,0.336722,18.879306,28.930713
4,oss,1.587314,0.621183,0.336722,13.395494,15.940713
